# 01 - EDA

Objectif: explorer rapidement les sources brutes (international results, FIFA rankings, groupes et calendrier).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"

def read_csv_safe(path: Path):
    if not path.exists():
        print(f"Missing file: {path}")
        return None
    return pd.read_csv(path)

intl_df = read_csv_safe(RAW_DIR / "international_results.csv")
rank_df = read_csv_safe(RAW_DIR / "fifa_rankings.csv")
groups_df = read_csv_safe(RAW_DIR / "wc2026_groups.csv")
schedule_df = read_csv_safe(RAW_DIR / "wc2026_schedule.csv")

Missing file: data/raw/international_results.csv
Missing file: data/raw/fifa_rankings.csv
Missing file: data/raw/wc2026_groups.csv
Missing file: data/raw/wc2026_schedule.csv


In [ ]:
if intl_df is not None:
    display(intl_df.head())
    print("international_results shape:", intl_df.shape)
    print(intl_df.columns)

if rank_df is not None:
    display(rank_df.head())
    print("fifa_rankings shape:", rank_df.shape)
    print(rank_df.columns)

if groups_df is not None:
    display(groups_df.head())
    print("groups shape:", groups_df.shape)

if schedule_df is not None:
    display(schedule_df.head())
    print("schedule shape:", schedule_df.shape)

In [3]:
if intl_df is not None:
    intl_df["date"] = pd.to_datetime(intl_df["date"], errors="coerce")
    outcome = np.where(
        intl_df["home_score"] > intl_df["away_score"],
        "W",
        np.where(intl_df["home_score"] < intl_df["away_score"], "L", "D")
    )
    outcome_counts = pd.Series(outcome).value_counts()
    print(outcome_counts)
    outcome_counts.plot(kind="bar", title="Home match outcome distribution")
    plt.show()

    tourn_counts = intl_df["tournament"].astype(str).str.lower().value_counts().head(10)
    print("Top tournaments:")
    print(tourn_counts)

In [4]:
if rank_df is not None and "rank" in rank_df.columns:
    rank_df["rank"] = pd.to_numeric(rank_df["rank"], errors="coerce")
    rank_df["rank"].dropna().plot(kind="hist", bins=40, title="FIFA rank distribution")
    plt.show()

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if intl_df is not None:
    outcome = np.where(
        intl_df["home_score"] > intl_df["away_score"],
        "W",
        np.where(intl_df["home_score"] < intl_df["away_score"], "L", "D")
    )
    outcome_counts = pd.Series(outcome).value_counts()
    ax = outcome_counts.plot(kind="bar", title="Home match outcome distribution")
    fig = ax.get_figure()
    fig.savefig(OUTPUT_DIR / "eda_outcome_distribution.png", dpi=150)
    outcome_counts.to_csv(OUTPUT_DIR / "eda_outcome_distribution.csv")
    plt.close(fig)

if rank_df is not None and "rank" in rank_df.columns:
    rank_df["rank"] = pd.to_numeric(rank_df["rank"], errors="coerce")
    ax = rank_df["rank"].dropna().plot(kind="hist", bins=40, title="FIFA rank distribution")
    fig = ax.get_figure()
    fig.savefig(OUTPUT_DIR / "eda_fifa_rank_distribution.png", dpi=150)
    plt.close(fig)